In [1]:
import pandas as pd 
import numpy as np 

np.random.seed(42)

In [2]:
# Generate advertisers

n_advertisers = 20

advertisers = pd.DataFrame({
    "advertiser_id": [
        f"ADV{i:03d}" for i in range(1, n_advertisers + 1)
    ],
    "advertiser_name": [
        f"Brand_{i}" for i in range(1, n_advertisers + 1)
    ],
    "industry": np.random.choice(["Fashion", "Beauty", "Electronics", "Sports"], n_advertisers)
})
advertisers.head()

,advertiser_id,advertiser_name,industry
0,ADV001,Brand_1,Electronics
1,ADV002,Brand_2,Sports
2,ADV003,Brand_3,Fashion
3,ADV004,Brand_4,Electronics
4,ADV005,Brand_5,Electronics


In [3]:
advertisers.shape

(20, 3)

In [4]:
# Generate products

n_products = 200

advertiser_ids = advertisers["advertiser_id"].unique()

# Create products
products = pd.DataFrame({
    "product_id": [
        f"PRDT_{i:03d}" for i in range(1, n_products + 1)
    ],
    "advertiser_id": np.random.choice(advertiser_ids, size=n_products)
})

# Add advertiser industry
products = products.merge(
    advertisers[["advertiser_id", "industry"]], on="advertiser_id", how="left"
)

# Define possible categories for each industry
categories = {
    "Sports": ["Running Shoes", "Sportswear", "Fitness Equipment"],
    "Beauty": ["Skincare", "Makeup", "Haircare"],
    "Electronics": ["Headphones", "Smartphones", "Laptops"],
    "Fashion": ["Clothing", "Shoes", "Accessories"]
}

# Function for choosing an appropriate category
def choose_category(industry):
    return np.random.choice(categories[industry])

# Give every product a category
products["category"] = products["industry"].apply(choose_category)

# Give every product a price
price_ranges = {
    "Running Shoes": (50, 200),
    "Sportswear": (20, 150),
    "Fitness Equipment": (30, 500),
    
    "Skincare": (10, 100),
    "Makeup": (5, 80),
    "Haircare": (8, 120),

    "Headphones": (20, 400),
    "Smartphones": (200, 1500),
    "Laptops": (400, 2500),

    "Clothing": (15, 250),
    "Shoes": (30, 300),
    "Accessories": (5, 200)
}

def choose_price(category):
    min_price, max_price = price_ranges[category]
    return round(np.random.uniform(min_price, max_price), 2)

products["price"] = products["category"].apply(choose_price)

In [5]:
products.head(10)

,product_id,advertiser_id,industry,category,price
0,PRDT_001,ADV002,Sports,Fitness Equipment,74.12
1,PRDT_002,ADV012,Electronics,Headphones,89.49
2,PRDT_003,ADV006,Sports,Running Shoes,190.19
3,PRDT_004,ADV002,Sports,Sportswear,102.98
4,PRDT_005,ADV001,Electronics,Laptops,1485.06
5,PRDT_006,ADV012,Electronics,Laptops,1779.93
6,PRDT_007,ADV012,Electronics,Smartphones,766.37
7,PRDT_008,ADV017,Sports,Sportswear,114.91
8,PRDT_009,ADV010,Beauty,Haircare,13.34
9,PRDT_010,ADV016,Fashion,Accessories,115.38


In [6]:
# Generate campaigns

n_campaigns = 50

campaigns = pd.DataFrame({
    "campaign_id": [
        f"CMP_{i:03d}" for i in range(1, n_campaigns + 1)
    ],
    "advertiser_id": np.random.choice(advertisers["advertiser_id"], size=n_campaigns),
    "daily_budget": np.random.uniform(100, 1000, size=n_campaigns).round(2),
    "bid": np.random.uniform(0.20, 2.00, size=n_campaigns).round(2)
})

In [7]:
campaigns.head()

,campaign_id,advertiser_id,daily_budget,bid
0,CMP_001,ADV017,912.84,1.07
1,CMP_002,ADV013,655.54,1.44
2,CMP_003,ADV001,982.42,1.12
3,CMP_004,ADV002,647.28,0.48
4,CMP_005,ADV009,672.98,0.88


In [8]:
# Generate customers

n_customers = 5000

customers = pd.DataFrame({
    "customer_id": [
        f"CUST_{i:05d}" for i in range(1, n_customers + 1)
    ],
    "country": np.random.choice(["Ghana", "Kenya", "Nigeria", "UK", "Germany"], size=n_customers),
    "device": np.random.choice(["Mobile", "Desktop", "Tablet"], size=n_customers),
    "customer_type": np.random.choice(["New", "Returning"], size=n_customers)
})

In [9]:
customers.head()

,customer_id,country,device,customer_type
0,CUST_00001,Germany,Mobile,New
1,CUST_00002,Ghana,Tablet,Returning
2,CUST_00003,Ghana,Tablet,New
3,CUST_00004,Germany,Mobile,Returning
4,CUST_00005,Germany,Tablet,Returning


In [10]:
customers.shape

(5000, 4)

In [11]:
# Generate ad_events

n_ad_events = 50000

ad_events = pd.DataFrame({
    "event_id": [f"EVT_{i:06d}" for i in range(1, n_ad_events + 1)],
    "customer_id": np.random.choice(customers["customer_id"], size=n_ad_events),
    "campaign_id": np.random.choice(campaigns["campaign_id"], size=n_ad_events)
})

# Find which advertiser owns which campaign
ad_events = ad_events.merge(
    campaigns[["campaign_id", "advertiser_id"]], on="campaign_id", how="left"
)

# Choose a product belonging to that advertiser
def choose_product(advertiser_id):
    advertiser_products = products[products["advertiser_id"] == advertiser_id]
    return np.random.choice(advertiser_products["product_id"])

ad_events["product_id"] = ad_events["advertiser_id"].apply(choose_product)

# Sponsored search position
ad_events["position"] = np.random.randint(1, 11, size=n_ad_events)

# Click probability by position
position_ctr = {
    1: 0.10,
    2: 0.08,
    3: 0.07,
    4: 0.06,
    5: 0.05,
    6: 0.04,
    7: 0.035,
    8: 0.03,
    9: 0.025,
    10: 0.02
}

def determine_click(position):
    click_probability = position_ctr[position]
    return np.random.choice([0, 1], p=[1- click_probability, click_probability])


ad_events["clicked"] = ad_events["position"].apply(determine_click)

# Event timestamp
start_date = pd.Timestamp("2026-06-01")

random_minutes = np.random.randint(0, 90 * 24 * 60, size=n_ad_events)

ad_events["event_time"] = (start_date + pd.to_timedelta(random_minutes, unit="m"))

ad_events = ad_events.sort_values("event_time").reset_index(drop=True)

ad_events.head(10)

,event_id,customer_id,campaign_id,advertiser_id,product_id,position,clicked,event_time
0,EVT_008726,CUST_00605,CMP_026,ADV013,PRDT_115,7,0,2026-06-01 00:00:00
1,EVT_011290,CUST_02215,CMP_026,ADV013,PRDT_063,5,0,2026-06-01 00:05:00
2,EVT_009506,CUST_02566,CMP_010,ADV017,PRDT_150,9,0,2026-06-01 00:06:00
3,EVT_040575,CUST_04936,CMP_041,ADV005,PRDT_113,6,0,2026-06-01 00:08:00
4,EVT_000159,CUST_03550,CMP_030,ADV017,PRDT_156,10,0,2026-06-01 00:08:00
5,EVT_012707,CUST_01007,CMP_020,ADV017,PRDT_036,2,0,2026-06-01 00:09:00
6,EVT_018799,CUST_02629,CMP_028,ADV004,PRDT_023,5,0,2026-06-01 00:12:00
7,EVT_013495,CUST_02454,CMP_038,ADV012,PRDT_068,7,0,2026-06-01 00:13:00
8,EVT_007593,CUST_03826,CMP_045,ADV015,PRDT_062,7,0,2026-06-01 00:13:00
9,EVT_038635,CUST_02346,CMP_048,ADV002,PRDT_001,1,1,2026-06-01 00:16:00


In [12]:
ad_events.shape

(50000, 8)

In [13]:
ad_events["clicked"].value_counts()

clicked
0    47486
1     2514
Name: count, dtype: int64

In [14]:
ad_events.groupby("position")["clicked"].mean()

position
1     0.097264
2     0.078462
3     0.070504
4     0.060624
5     0.051507
6     0.038885
7     0.033367
8     0.028246
9     0.023166
10    0.019435
Name: clicked, dtype: float64

In [15]:
# Generate dataframe of clicked ad_events only

clicks = ad_events[ad_events["clicked"] == 1].copy()

In [16]:
# Generate orders

n_orders = 3000

# number of orders bought because they clicked on ad
n_ad_influenced = 600
# number of orders purchased without any ad influence
n_organic = 2400

selected_clicks = clicks.sample(n=n_ad_influenced, replace=True, random_state=42).copy()

ad_orders = pd.DataFrame({
    "customer_id": selected_clicks["customer_id"].values,
    "product_id": selected_clicks["product_id"].values
})

# Add delay to simulate customers purchasing item between 1 to 7 days after they first saw it
purchase_delay_hours = np.random.randint(1, 7 * 24 + 1, size=n_ad_influenced)

ad_orders["order_time"] = (selected_clicks["event_time"].values + pd.to_timedelta(purchase_delay_hours, unit="h"))

# create organic orders
organic_orders = pd.DataFrame({
    "customer_id": np.random.choice(customers["customer_id"], size=n_organic),
    "product_id": np.random.choice(products["product_id"], size=n_organic)
})
organic_orders["order_time"] = (start_date + pd.to_timedelta(np.random.randint(0, 90*24*60, size=n_organic), unit="m"))

# Join organic and ad generated orders together
orders = pd.concat([organic_orders, ad_orders], ignore_index=True)
orders["order_id"] = [f"ORD_{i:06d}" for i in range(1, len(orders) + 1)]

# Merge product price from product dataframe to orders and rename as revenue
orders = orders.merge(products[["product_id", "price"]], on="product_id", how="left")
orders = orders.rename(columns={"price": "revenue"})

In [17]:
ad_events.head()

,event_id,customer_id,campaign_id,advertiser_id,product_id,position,clicked,event_time
0,EVT_008726,CUST_00605,CMP_026,ADV013,PRDT_115,7,0,2026-06-01 00:00:00
1,EVT_011290,CUST_02215,CMP_026,ADV013,PRDT_063,5,0,2026-06-01 00:05:00
2,EVT_009506,CUST_02566,CMP_010,ADV017,PRDT_150,9,0,2026-06-01 00:06:00
3,EVT_040575,CUST_04936,CMP_041,ADV005,PRDT_113,6,0,2026-06-01 00:08:00
4,EVT_000159,CUST_03550,CMP_030,ADV017,PRDT_156,10,0,2026-06-01 00:08:00


In [18]:
ad_events["clicked"].value_counts()

clicked
0    47486
1     2514
Name: count, dtype: int64

In [19]:
ad_events.groupby("position")["clicked"].mean()

position
1     0.097264
2     0.078462
3     0.070504
4     0.060624
5     0.051507
6     0.038885
7     0.033367
8     0.028246
9     0.023166
10    0.019435
Name: clicked, dtype: float64

In [20]:
# Attribution function: to associate ad-driven orders to their campaign, event_id
# If there are multiple ad impressions over multiple days, choose the last date the user say the ad

"""
User saw ad for PRDT_072 and clicked on it in on 2026-06-25, 2026-06-30, and 2026-07-02.
In all 3 instances they did not purchase the product.

On 2026-07-04, the customer bought the product via order id ORD_002401.
Based on 7-day attribution strategy, the last click date (2026-07-02 19:49:00) is credited to be the date that influenced the purchase.
"""

def attribute_order(order):
    customer = order["customer_id"]
    product = order["product_id"]
    purchase_time = order["order_time"]

    window_start = purchase_time - pd.Timedelta(days=7)

    eligible_clicks = clicks[
        (clicks["customer_id"] == customer) & (clicks["product_id"] == product) &
        (clicks["event_time"] >= window_start) & (clicks["event_time"] <= purchase_time)
    ]

    if eligible_clicks.empty:
        return pd.Series({
            "attributed_to_ad": 0,
            "attributed_campaign_id": None,
            "attributed_event_id": None,
            "click_time": pd.NaT,
            "days_since_click": np.nan
        })

    last_click = eligible_clicks.sort_values("event_time").iloc[-1]
    time_difference = (purchase_time - last_click["event_time"])

    return pd.Series({
        "attributed_to_ad": 1,
        "attributed_campaign_id": last_click["campaign_id"],
        "attributed_event_id": last_click["event_id"],
        "click_time": last_click["event_time"],
        "days_since_click": round(time_difference.total_seconds() / 86400, 2)
    })

In [21]:
attribution_results = orders.apply(attribute_order, axis=1)
attribution_results.head()

,attributed_to_ad,attributed_campaign_id,attributed_event_id,click_time,days_since_click
0,0,NaN,NaN,NaT,NaN
1,0,NaN,NaN,NaT,NaN
2,0,NaN,NaN,NaT,NaN
3,0,NaN,NaN,NaT,NaN
4,0,NaN,NaN,NaT,NaN


In [22]:
orders = pd.concat([orders, attribution_results], axis=1)
orders.head()

,customer_id,product_id,order_time,order_id,revenue,attributed_to_ad,attributed_campaign_id,attributed_event_id,click_time,days_since_click
0,CUST_04201,PRDT_133,2026-06-15 11:44:00,ORD_000001,523.34,0,NaN,NaN,NaT,NaN
1,CUST_00165,PRDT_161,2026-06-30 18:02:00,ORD_000002,131.90,0,NaN,NaN,NaT,NaN
2,CUST_00851,PRDT_122,2026-07-06 11:45:00,ORD_000003,303.21,0,NaN,NaN,NaT,NaN
3,CUST_00023,PRDT_160,2026-07-06 08:05:00,ORD_000004,41.61,0,NaN,NaN,NaT,NaN
4,CUST_01749,PRDT_156,2026-07-07 01:11:00,ORD_000005,139.49,0,NaN,NaN,NaT,NaN


In [23]:
orders["attributed_to_ad"].value_counts()

attributed_to_ad
0    2400
1     600
Name: count, dtype: int64

In [24]:
attribution_rate = orders["attributed_to_ad"].mean()
print(f"Attribution rate: {attribution_rate:.2%}")

Attribution rate: 20.00%


In [25]:
orders[
    orders["attributed_to_ad"] == 1
][
    [
        "order_id", "customer_id", "product_id", "order_time", "attributed_campaign_id", "click_time", "days_since_click"
    ]
]

,order_id,customer_id,product_id,order_time,attributed_campaign_id,click_time,days_since_click
2400,ORD_002401,CUST_01911,PRDT_072,2026-07-04 08:49:00,CMP_010,2026-07-02 19:49:00,1.54
2401,ORD_002402,CUST_02140,PRDT_020,2026-07-24 12:40:00,CMP_022,2026-07-17 22:40:00,6.58
2402,ORD_002403,CUST_02969,PRDT_132,2026-07-17 14:42:00,CMP_024,2026-07-12 08:42:00,5.25
2403,ORD_002404,CUST_02346,PRDT_140,2026-07-14 04:22:00,CMP_005,2026-07-11 00:22:00,3.17
2404,ORD_002405,CUST_03027,PRDT_094,2026-08-01 20:25:00,CMP_014,2026-07-29 03:25:00,3.71
...,...,...,...,...,...,...,...
2995,ORD_002996,CUST_04088,PRDT_040,2026-07-12 03:23:00,CMP_028,2026-07-05 04:23:00,6.96
2996,ORD_002997,CUST_04383,PRDT_060,2026-08-13 20:18:00,CMP_002,2026-08-12 08:18:00,1.50
2997,ORD_002998,CUST_01765,PRDT_013,2026-09-05 09:34:00,CMP_025,2026-08-29 18:34:00,6.62
2998,ORD_002999,CUST_04216,PRDT_189,2026-07-11 15:27:00,CMP_006,2026-07-05 02:27:00,6.54


In [26]:
# Already generated and saved into dataset folder

"""
advertisers.to_csv("../data/raw/advertisers.csv", index=False)
products.to_csv("../data/raw/products.csv", index=False)
customers.to_csv("../data/raw/customers.csv", index=False)
campaigns.to_csv("../data/raw/campaigns.csv", index=False)
ad_events.to_csv("../data/raw/ad_events.csv", index=False)
orders.to_csv("../data/raw/orders.csv", index=False)
"""

'\nadvertisers.to_csv("../data/raw/advertisers.csv", index=False)\nproducts.to_csv("../data/raw/products.csv", index=False)\ncustomers.to_csv("../data/raw/customers.csv", index=False)\ncampaigns.to_csv("../data/raw/campaigns.csv", index=False)\nad_events.to_csv("../data/raw/ad_events.csv", index=False)\norders.to_csv("../data/raw/orders.csv", index=False)\n'